# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abc085455-byte/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signal checks first — the rule can only lean on what actually shows up.**

- **Signal A — staleness (`freshness_tier` / `days_since_last_update`).** This is the signal behind FlyRank's real refresh flags (`stale_visible_page`, the refresh-tier logic): the story is "pages that haven't been touched in a long time decline more." Bucket table below, by `freshness_tier`, decline rate = share with `trend_direction == "down"`, n printed per bucket.
- **Signal B — CTR vs. position (`ctr` by `position_tier`).** This is the signal behind the real CTR-fix logic (`low_ctr_visible_page`): the story is "CTR depends heavily on position, so a page's CTR only means something compared to its own tier." Bucket table below, by `position_tier`, restricted to pages with real position data and enough volume (`impressions_90d >= 100`) so a floor artifact doesn't masquerade as a finding — the data dictionary warns the raw `top_3` bucket has a median volume of ~3 impressions, cheap enough for one lucky/unlucky click to swing the whole median.

Verdicts use the session's four words: **CONFIRMED / OPPOSITE / MIXED / FALSE.**


In [ ]:
import pandas as pd, numpy as np, os

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
overall_decline_rate = df["is_declining_label"].mean()
print(f"rows: {len(df)} | overall decline rate: {overall_decline_rate:.3f}")

# --- Signal A: staleness (freshness_tier) vs. decline rate ---
tier_order = ["0-30", "31-90", "91-180", "181+"]
signal_a = df.groupby("freshness_tier")["is_declining_label"].agg(
    decline_rate="mean", n="count"
).reindex(tier_order)
print("\nSignal A - staleness (freshness_tier) vs. decline rate:")
print(signal_a)


rows: 30000 | overall decline rate: 0.542

Signal A - staleness (freshness_tier) vs. decline rate:
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
31-90               0.588571    175
91-180              0.611057   9171
181+                0.471264    174


**Verdict — Signal A (staleness): MIXED.**

Decline rate is *not* monotonic in staleness: `0-30` = 51.1% (n=20,480), `31-90` = 58.9%
(n=175, a thin cell — read cautiously), `91-180` = 61.1% (n=9,171), and the very stalest bucket,
`181+` = 47.1% (n=174) — actually *below* the overall 54.2% rate. If staleness alone drove
decline the way the refresh-flag story assumes, `181+` should be the highest bar, not the
lowest. It isn't. This is a clearly-explained negative, not a shrug: on this slice, raw
"hasn't been touched in N days" alone does not reliably predict "currently trending down."
That's a real finding, and it means my rule should not lean on staleness by itself as the
main driver — a rule built purely on `days_since_last_update >= 180` would be riding a bucket
that the data itself contradicts.


In [ ]:
# --- Signal B: CTR vs. position tier, with a volume floor so a thin bucket ---
# --- doesn't fake a median (per the data dictionary's own warning on top_3) ---
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
visible_enough = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]

signal_b = visible_enough.groupby("position_tier")["ctr"].agg(
    median_ctr="median", mean_ctr="mean", n="count"
).reindex(pos_order)
print("Signal B - CTR by position_tier (impressions_90d >= 100, avg_position > 0):")
print(signal_b)


Signal B - CTR by position_tier (impressions_90d >= 100, avg_position > 0):
               median_ctr  mean_ctr     n
position_tier                            
top_3                0.19  0.334128   533
page_1               0.23  0.354760  8633
striking             0.15  0.255782  5903
page_3_5             0.06  0.142359  6058
deep                 0.00  0.055415   879


**Verdict — Signal B (CTR vs. position): CONFIRMED, with one caveat.**

Median CTR clearly tracks position quality once low-volume noise is filtered out: `page_1`
0.23% (n=8,633) > `striking` 0.15% (n=5,903) > `page_3_5` 0.06% (n=6,058) > `deep` 0.00%
(n=879). The `top_3` bucket (0.19%, n=533) sits just under `page_1` instead of on top — a small,
plausible wobble given it's the thinnest tier here, not a reversal of the story. The practical
takeaway holds: **a raw CTR number means nothing on its own — it only means something compared
to other pages in the same position tier.** That's exactly the comparison FlyRank's CTR-fix
logic needs, and exactly what my rule below will use instead of a single fixed CTR threshold.

**My rule, in plain words:** *A page is worth reviewing for a CTR/title fix if it's getting
real search demand (visible: `impressions_90d >= 500`, with `avg_position > 0` so
"no position data" rows can't fake their way into a tier), AND its CTR is below the
median CTR of other visible pages that share its position tier — i.e. it's underperforming
its peers, not just underperforming some fixed number. Rank the flagged pages by how much
exposure they have, since a low-CTR page with 500,000 impressions matters far more than one
with 500.* I'm deliberately **not** folding staleness into the score — Signal A just showed it
doesn't confirm here, and encoding a signal I've already watched fail would make the rule less
honest, not more sophisticated.

**Reason code (one):** `low_ctr_visible_page` — visible, real position, CTR below its own
tier's median. Non-flagged pages get reason code `none`.

**Action label:** `review_ctr_fix` for flagged pages, `monitor` for everyone else.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = `impressions_90d` when the page is visible, has real position data, and its CTR sits
below its own position tier's median CTR — else `0`. No fitted weights, no future-window
columns, nothing derived from the label (`trend_direction` / `trend_pct` never touched).


In [ ]:
# --- The rule, coded exactly as described above ---
visible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0)

# Tier medians computed ONLY on the visible population -- this is a same-row-window
# comparison group, not a future or label-derived number.
tier_median_ctr = df.loc[visible].groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)

ctr_gap = visible & (df["ctr"] < df["tier_median_ctr"])

df["baseline_score"] = np.where(ctr_gap, df["impressions_90d"], 0)
df["reason_code"] = np.where(ctr_gap, "low_ctr_visible_page", "none")
df["action"] = np.where(ctr_gap, "review_ctr_fix", "monitor")

print("visible (impressions_90d>=500, avg_position>0):", int(visible.sum()))
print("flagged low_ctr_visible_page:", int(ctr_gap.sum()),
      f"({100*ctr_gap.mean():.1f}% of all rows)")

# Honest sanity check against the proxy label -- NOT used to build the score, only to look at it
base_rate = df["is_declining_label"].mean()
flagged_rate = df.loc[ctr_gap, "is_declining_label"].mean()
print(f"\nBase decline rate (all rows): {base_rate:.3f}")
print(f"Decline rate among flagged rows: {flagged_rate:.3f}  (context only -- CTR-fix isn't trying to predict 'decline')")

ranked = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

out_cols = [
    "rank", "content_id", "client_id", "impressions_90d", "avg_position", "position_tier",
    "ctr", "tier_median_ctr", "word_count", "content_type", "baseline_score",
    "reason_code", "action",
]

os.makedirs("../outputs", exist_ok=True)
out_path = "../outputs/baseline_action_score.csv"
ranked[out_cols].to_csv(out_path, index=False)
print(f"\nWrote {len(ranked)} ranked rows to {out_path}")
ranked[out_cols].head(10)


visible (impressions_90d>=500, avg_position>0): 16726
flagged low_ctr_visible_page: 7950 (26.5% of all rows)

Base decline rate (all rows): 0.542
Decline rate among flagged rows: 0.656  (context only -- CTR-fix isn't trying to predict 'decline')



Wrote 30000 ranked rows to ../outputs/baseline_action_score.csv


,rank,content_id,client_id,impressions_90d,avg_position,position_tier,ctr,tier_median_ctr,word_count,content_type,baseline_score,reason_code,action
0,1,content_5fe46e04994d,client_4e07408562,517715,4.2,page_1,0.14,0.24,NaN,keyword article,517715,low_ctr_visible_page,review_ctr_fix
1,2,content_8c19996aa890,client_4e07408562,509252,2.5,top_3,0.15,0.20,2895.0,keyword article,509252,low_ctr_visible_page,review_ctr_fix
2,3,content_1a9e894be2e2,client_19581e27de,416180,4.0,page_1,0.23,0.24,NaN,keyword article,416180,low_ctr_visible_page,review_ctr_fix
3,4,content_db5989a78dd3,client_4e07408562,345111,5.4,page_1,0.21,0.24,2682.0,keyword article,345111,low_ctr_visible_page,review_ctr_fix
4,5,content_cb112fce36be,client_19581e27de,309910,5.6,page_1,0.16,0.24,2761.0,keyword article,309910,low_ctr_visible_page,review_ctr_fix
5,6,content_36ff89c8214e,client_19581e27de,295097,7.3,page_1,0.05,0.24,NaN,keyword article,295097,low_ctr_visible_page,review_ctr_fix
6,7,content_b28d1efd668f,client_6208ef0f77,286608,26.2,page_3_5,0.06,0.09,6901.0,keyword article,286608,low_ctr_visible_page,review_ctr_fix
7,8,content_8451fc6f034d,client_d029fa3a95,272144,2.3,top_3,0.03,0.20,3528.0,keyword article,272144,low_ctr_visible_page,review_ctr_fix
8,9,content_813e88069237,client_6208ef0f77,233561,26.2,page_3_5,0.06,0.09,4610.0,keyword article,233561,low_ctr_visible_page,review_ctr_fix
9,10,content_ff94c9b6b411,client_349c41201b,228566,27.4,page_3_5,0.04,0.09,5375.0,keyword article,228566,low_ctr_visible_page,review_ctr_fix


## 3. Top-10 review

*For each of your top ten, one line each — the action, why it's there, and what would make it wrong.*


In [ ]:
top10 = ranked.head(10)
print(top10[["rank", "client_id", "impressions_90d", "avg_position", "position_tier",
             "ctr", "tier_median_ctr", "content_type"]].to_string(index=False))


 rank         client_id  impressions_90d  avg_position position_tier  ctr  tier_median_ctr    content_type
    1 client_4e07408562           517715           4.2        page_1 0.14             0.24 keyword article
    2 client_4e07408562           509252           2.5         top_3 0.15             0.20 keyword article
    3 client_19581e27de           416180           4.0        page_1 0.23             0.24 keyword article
    4 client_4e07408562           345111           5.4        page_1 0.21             0.24 keyword article
    5 client_19581e27de           309910           5.6        page_1 0.16             0.24 keyword article
    6 client_19581e27de           295097           7.3        page_1 0.05             0.24 keyword article
    7 client_6208ef0f77           286608          26.2      page_3_5 0.06             0.09 keyword article
    8 client_d029fa3a95           272144           2.3         top_3 0.03             0.20 keyword article
    9 client_6208ef0f77           233

**Top-10 review** (all ten carry action `review_ctr_fix`, reason code `low_ctr_visible_page`):

1. **Rank 1** — 517,715 impressions, position 4.2 (`page_1`), CTR 0.14% vs. tier median 0.24%.
   Here because it's the single biggest exposure gap in the dataset. Wrong if this page's
   audience is mostly navigational/branded searches, where low CTR is normal, not a title problem.
2. **Rank 2** — 509,252 impressions, position 2.5 (`top_3`), CTR 0.15% vs. tier median 0.20%.
   Here for the same reason at the very top of the SERP. Wrong if a featured snippet or a
   People-Also-Ask box above it is siphoning clicks — that's a SERP-layout issue, not a title fix.
3. **Rank 3** — 416,180 impressions, position 4.0 (`page_1`), CTR 0.23% vs. tier median 0.24%.
   Here on volume, but the CTR gap is tiny (0.01pp) — near the tier median, not badly below it.
   Wrong if this is really just normal peer variation and not an actionable problem at all.
4. **Rank 4** — 345,111 impressions, position 5.4 (`page_1`), CTR 0.21% vs. tier median 0.24%.
   Here for a moderate, real gap on high volume. Wrong if the query's intent is informational and
   the SERP already shows an AI overview that answers it without a click.
5. **Rank 5** — 309,910 impressions, position 5.6 (`page_1`), CTR 0.16% vs. tier median 0.24%.
   A clear gap, high volume — good candidate. Wrong if it's a recent page still stabilizing
   (position/CTR can be noisy in a page's first weeks, and we don't have an age check on this row).
6. **Rank 6** — 295,097 impressions, position 7.3 (`page_1`), CTR 0.05% vs. tier median 0.24%.
   The widest CTR gap in the top 10 relative to its tier — strong candidate. Wrong if the title
   already matches intent and the real issue is a weak or missing meta description/rich snippet.
7. **Rank 7** — 286,608 impressions, position 26.2 (`page_3_5`), CTR 0.06% vs. tier median 0.09%.
   Here on sheer volume even at a weaker position. Wrong if position 26 genuinely bounds CTR here —
   i.e., the page is already doing about as well as any `page_3_5` page can, and a title rewrite
   won't move a metric that's really capped by rank, not by wording.
8. **Rank 8** — 272,144 impressions, position 2.3 (`top_3`), CTR 0.03% vs. tier median 0.20%.
   The single largest relative gap in the top 10 (CTR is ~1/7th of its tier's median at a
   near-top-3 position) — the highest-confidence pick on this list. Wrong if this is a
   client-specific tracking anomaly (e.g. impressions logged but a redirect eating clicks).
9. **Rank 9** — 233,561 impressions, position 26.2 (`page_3_5`), CTR 0.06% vs. tier median 0.09%.
   Same client and tier as rank 7 — here on volume and a real, if modest, gap. Wrong if this and
   rank 7 are near-duplicate/sibling pages competing for the same query (cannibalization), in
   which case the fix is consolidation, not a title/meta edit on either page alone.
10. **Rank 10** — 228,566 impressions, position 27.4 (`page_3_5`), CTR 0.04% vs. tier median
    0.09%. Here on volume and gap size. Wrong if `page_3_5` at this client is systematically
    dragged down by one outlier query in the mix rather than a page-level problem I can fix.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks, by hand:**

- **Rank 3** is the weakest of the ten: its CTR (0.23%) is only 0.01 percentage points under
  its tier median (0.24%) — well within the kind of peer-to-peer noise Signal B's own bucket
  table shows (`page_1`'s CTR values spread well beyond a 0.01pp band). It only made the top 10
  because `impressions_90d` is huge; the *gap* itself barely clears zero. A stricter version of
  this rule would require the gap to be some minimum percentage of the tier median, not just
  any gap above zero, before it counts as a real underperformer.
- **Ranks 7 and 9** share a client and a tier (`page_3_5`, `client_6208ef0f77`) with very
  similar CTR and gap sizes. That's a real concentration risk in a "review the top 10" queue —
  one client is quietly occupying 20% of the list, and if they're actually cannibalizing each
  other (two pages splitting one query's demand) the honest fix isn't "review both titles," it's
  "check whether these should be one page."
- A found data-quality issue that *would* have broken this rule if I'd skipped the check: 1,205
  rows have `avg_position == 0`, which the dictionary says means "no position data" — but every
  one of them still lands in `position_tier == "top_3"` (0 ≤ 3). Without the explicit
  `avg_position > 0` filter in the score, those no-data rows would have been silently compared
  against real top-3 pages and could have entered the queue on a fake tier assignment.

**Leakage checklist:**

- `trend_direction` and `trend_pct` (the label source) are used only to *report* the proxy
  decline rate for context in Section 2 — never inside `baseline_score`, `ctr_gap`, or
  `tier_median_ctr`.
- No FlyRank product flags (`health_score`, `priority_score`, `action_type`, `needs_ctr_fix`,
  `is_quick_win`) are in this starter dataset at all, so there's nothing to accidentally
  re-use as an input — the rule is built entirely from observable 90-day signals
  (`impressions_90d`, `avg_position`, `position_tier`, `ctr`) that were knowable before any
  decision was made.
- `tier_median_ctr` is computed on the *same* trailing 90-day window as every feature that feeds
  it — no future window, no window overlap, no per-row lookahead.
- `content_id` / `client_id` are used for display and grouping only, never as score inputs.


In [ ]:
# Verify the rank-3 gap-size claim
rank3 = ranked.iloc[2]
gap_pp = rank3["tier_median_ctr"] - rank3["ctr"]
print(f"Rank 3 gap: {gap_pp:.3f} percentage points (ctr={rank3['ctr']}, tier_median={rank3['tier_median_ctr']})")

# Verify the client-concentration claim in ranks 7 & 9
print("\nTop-10 client counts:")
print(ranked.head(10)["client_id"].value_counts())

# Verify the avg_position == 0 / top_3 miscategorization data-quality issue
bad_top3 = df[(df["avg_position"] == 0)]
print(f"\navg_position == 0 rows: {len(bad_top3)} | all mapped to position_tier == 'top_3': "
      f"{(bad_top3['position_tier'] == 'top_3').all()}")

# Confirm the label columns never touch the score
label_cols_untouched = not any(c in ["trend_direction", "trend_pct"]
                                for c in ["impressions_90d", "avg_position", "position_tier", "ctr"])
print(f"\nScore built only from observable signals (no label columns among inputs): {label_cols_untouched}")


Rank 3 gap: 0.010 percentage points (ctr=0.23, tier_median=0.24)

Top-10 client counts:
client_id
client_4e07408562    3
client_19581e27de    3
client_6208ef0f77    2
client_d029fa3a95    1
client_349c41201b    1
Name: count, dtype: int64

avg_position == 0 rows: 1205 | all mapped to position_tier == 'top_3': True

Score built only from observable signals (no label columns among inputs): True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.